In [25]:
import os

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_ollama import ChatOllama
from langchain_ollama import OllamaEmbeddings

from langchain_chroma import Chroma

from langchain_core.stores import InMemoryStore

from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

from langchain_core.prompts import ChatPromptTemplate

In [26]:
loader = PyPDFLoader("dataset/company_policy.pdf")
docs = loader.load()

print("Pages:", len(docs))

Pages: 10


In [27]:
parent_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=200
)

In [28]:
child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=50
)

In [29]:
embeddings = OllamaEmbeddings(
    model="nomic-embed-text"
)

In [30]:
vectorstore = Chroma(
    collection_name="parent_documents",
    embedding_function=embeddings,
    persist_directory="chroma_parent_db"
)

In [31]:
store = InMemoryStore()

In [32]:
retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)

In [34]:
retriever.add_documents(docs)

print("Parent Chunks:", len(list(store.yield_keys())))

Parent Chunks: 18


In [35]:
llm = ChatOllama(
    model="mistral",
    temperature=0
)

In [36]:
prompt = ChatPromptTemplate.from_template("""
You are a helpful AI assistant.

Answer ONLY using the provided context.

If the answer is not available in the context,
reply:

"I don't know based on the provided documents."

Context:
{context}

Question:
{input}
""")

In [37]:
document_chain = create_stuff_documents_chain(
    llm,
    prompt
)

retrieval_chain = create_retrieval_chain(
    retriever,
    document_chain
)

In [38]:
query = "What is the policy on remote work setup allowance?"

response = retrieval_chain.invoke(
    {"input": query}
)

print("\nAnswer:\n")
print(response["answer"])


Answer:

 I don't know based on the provided documents. The context does not mention any policy on remote work setup allowance.


In [39]:
print("\nRetrieved Parent Chunks:\n")

for i, doc in enumerate(response["context"], 1):
    print("=" * 80)
    print(f"Parent Chunk {i}")
    print("=" * 80)
    print(doc.page_content[:1000])
    print()


Retrieved Parent Chunks:

Parent Chunk 1
lunch
 
break.
  **5.2  Flexible  Work  Arrangements**  Subject  to  business  requirements  and  Manager  approval,  the  Company  may  offer  hybrid  
working
 
options
 
(e.g.,
 
work
 
from
 
home).
 
Employees
 
working
 
remotely
 
must
 
be
 
available
 
on
 
all
 
official
 
communication
 
channels
 
during
 
core
 
business
 
hours
 
(11:00
 
AM
 
to
 
4:00
 
PM).
 
Ad-hoc
 
remote
 
work
 
requests
 
must
 
be
 
approved
 
via
 
email
 
24
 
hours
 
in
 
advance.
  **5.3  Attendance  &  Biometrics**  All  employees  are  required  to  record  their  attendance  using  the  biometric  system  or  the  
designated
 
digital
 
attendance
 
application.
 
Manual
 
registers
 
will
 
only
 
be
 
used
 
in
 
the
 
event
 
of
 
a
 
system
 
failure.
 
Failure
 
to
 
mark
 
attendance
 
correctly
 
may
 
result
 
in
 
a
 
salary
 
deduction
 
for
 
that
 
day.
  **5.4  Late  Coming  &  Half-Day**

Parent Chunk 2
**7.4  Reimbursements**  -  *